# RoofTop Regression Analysis

## Purpose

This notebook investigates a **regression in rooftop metrics** for APT (Address Point) data.
A regression here means rooftop accuracy has dropped between two snapshots, and the most
likely cause is that some APTs present in the **old snapshot** are **missing from the latest
snapshot**.

## Approach

The first step is to identify **which APTs went missing** between the two snapshots:

1. **Load the old snapshot** into a Delta table.
2. **Load the latest snapshot** into a Delta table.
3. **Find the missing APTs** — the APTs present in the old snapshot but absent from the
   latest snapshot (a `leftanti` join on the APT `unsigned_id`).

The set of missing APTs is the input for the downstream rooftop regression investigation.


In [ ]:
%scala
// ============================================================
// Load OLD snapshot -> Delta table
// Pin OLD_REVISION to the older revision you want to compare against.
// Saved to: my_database.apt_snapshot_revision_{OLD_REVISION}_14533
// ============================================================

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository

val LAYER_ID = 14533

// >>> Set the OLD snapshot revision id here <<<
val OLD_REVISION = 42037615L

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
println(s"Environment Name $env")

implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)
println(ConfigLoader.getConfig.getString("secrets.key-vault.url"))

println(s"Loading OLD snapshot revision: $OLD_REVISION")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, OLD_REVISION)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val config = SourceConfigLoader.getConfig
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")

SparkHelper.init("data_recovery")
new LoadFreshSnapshotData().run()

// Persist the loaded old snapshot into a Delta table
val oldSnapshotTable = s"my_database.apt_snapshot_revision_${OLD_REVISION}_${LAYER_ID}"
val oldAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
oldAptDS.write.format("delta").mode("overwrite").saveAsTable(oldSnapshotTable)
println(s"OLD snapshot written to: $oldSnapshotTable")
display(oldAptDS)


In [ ]:
%scala
// ============================================================
// Load LATEST snapshot -> Delta table
// Uses LoadSnapshotInfo.getLatestRevisionId to resolve the newest revision.
// Saved to: my_database.apt_snapshot_revision_{latestRevision}_14533
// ============================================================

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import org.apache.sedona.spark.SedonaContext
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository

val LAYER_ID = 14533

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
println(s"Environment Name $env")

implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)
println(ConfigLoader.getConfig.getString("secrets.key-vault.url"))

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

val config = SourceConfigLoader.getConfig
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(databaseName, tableName) = deltaConfig.split("\\.")
println(s"Database Name: $databaseName")
println(s"Table Name: $tableName")

SparkHelper.init("data_recovery")
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot into a Delta table
val latestSnapshotTable = s"my_database.apt_snapshot_revision_${latestRevision.get}_${LAYER_ID}"
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")
display(latestAptDS)


In [ ]:
%scala
// ============================================================
// Read OLD snapshot Delta table as Dataset[OrbisElement]
// then enrich with `country` and `orbisIdString` columns.
// ============================================================

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.{Id, OrbisElement}
import java.lang.{Long => JLong}

// Build the colon-separated unsigned id string: {layerId}:{unsignedHigh}:{unsignedLow}
def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"

val oldSnapshotDS: Dataset[OrbisElement] =
  spark.table(oldSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val oldSnapshotEnriched = oldSnapshotDS
  // country = value of the metadata:country tag
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  // orbisIdString = unsigned colon-separated id
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"OLD snapshot read from: $oldSnapshotTable, count: ${oldSnapshotEnriched.count()}")
display(oldSnapshotEnriched)


In [ ]:
%scala
// ============================================================
// Read LATEST snapshot Delta table as Dataset[OrbisElement]
// then enrich with `country` and `orbisIdString` columns.
// (convertOrbisIdUDF and countryTagKey are defined in the OLD-read cell.)
// ============================================================

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.OrbisElement

val latestSnapshotDS: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val latestSnapshotEnriched = latestSnapshotDS
  // country = value of the metadata:country tag
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  // orbisIdString = unsigned colon-separated id
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"LATEST snapshot read from: $latestSnapshotTable, count: ${latestSnapshotEnriched.count()}")
display(latestSnapshotEnriched)
